In [ ]:
!pip install requests pandas python-dotenv

In [ ]:
import requests # permet de faire les requetes HTTP
import json # Permet de lire et écrire des fichiers JSON
import os #on lit la cle API 
from dotenv import load_dotenv #permet de lire les fichiers .env
import pandas 


#charger la cle depuis .env
load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")
print ("clé trouvable " if API_KEY else "ERREUR : CLE NON TROUVABLE ")


In [ ]:
# ID de la chaîne YouTube - MrBeast
CHANNEL_ID = "UCX6OQ3DkcsbYNE6H8uQQuVA"
print(f"Chaîne sélectionnée : {CHANNEL_ID}")

In [ ]:
# Cette fonction récupère la playlist principale de la chaîne
def get_uploads_playlist(channel_id, api_key):
    
    # L'adresse de l'API YouTube pour les chaînes
    url = "https://www.googleapis.com/youtube/v3/channels"
    
    # Les paramètres qu'on envoie à l'API
    params = {
        "part": "contentDetails",  # On veut les détails de la chaîne
        "id": channel_id,          # L'ID de MrBeast
        "key": api_key             # Notre clé API pour s'authentifier
    }
    
    # On envoie la requête à YouTube
    response = requests.get(url, params=params)
    
    # On convertit la réponse en dictionnaire Python
    data = response.json()
    
    # On extrait l'ID de la playlist principale
    playlist_id = data["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
    
    print(f"Playlist trouvée : {playlist_id} ✅")
    return playlist_id

# On appelle la fonction avec MrBeast
playlist_id = get_uploads_playlist(CHANNEL_ID, API_KEY)

In [6]:
url = "https://www.googleapis.com/youtube/v3/playlistItems"

video_ids = []
next_page_token = None

# Pas de limite → on récupère TOUT
while True:
    params = {
        "part": "contentDetails",
        "playlistId": playlist_id,
        "maxResults": 50,
        "key": API_KEY,
        "pageToken": next_page_token
    }

    response = requests.get(url, params=params)
    data = response.json()

    for item in data["items"]:
        video_ids.append(item["contentDetails"]["videoId"])

    next_page_token = data.get("nextPageToken")
    
    # S'il n'y a plus de page → STOP
    if not next_page_token:
        break

print(f"{len(video_ids)} vidéos trouvées ✅")

970 vidéos trouvées ✅


In [ ]:
# L'adresse de l'API YouTube pour les vidéos
url = "https://www.googleapis.com/youtube/v3/videos"

# Liste vide pour stocker toutes les vidéos
videos = []

# On traite les 968 vidéos par lots de 50
# car Google accepte max 50 par requête
for i in range(0, len(video_ids), 50):
    
    # On prend 50 IDs à la fois
    batch = video_ids[i:i+50]
    
    # Les paramètres qu'on envoie à YouTube
    params = {
        "part": "snippet,contentDetails,statistics", # On veut tout
        "id": ",".join(batch),                       # Les 50 IDs
        "key": API_KEY                               # Notre clé API
    }
    
    # On envoie la requête à YouTube
    response = requests.get(url, params=params)
    
    # On convertit la réponse en dictionnaire Python
    data = response.json()
    
    # Pour chaque vidéo on extrait les infos
    for item in data["items"]:
        video = {
            "videoId"      : item["id"],                              # ID
            "title"        : item["snippet"]["title"],                # Titre
            "publishedAt"  : item["snippet"]["publishedAt"],          # Date
            "duration"     : item["contentDetails"]["duration"],      # Durée
            "viewCount"    : item["statistics"].get("viewCount", 0),  # Vues
            "likeCount"    : item["statistics"].get("likeCount", 0),  # Likes
            "commentCount" : item["statistics"].get("commentCount", 0)# Commentaires
        }
        # On ajoute la vidéo dans la liste
        videos.append(video)

# On affiche le total
print(f"{len(videos)} vidéos extraites ")

In [ ]:
import os

# Créer le dossier data/raw
os.makedirs("../data/raw", exist_ok=True)

# Sauvegarder les données en JSON
with open("../data/raw/videos_raw.json", "w", encoding="utf-8") as f:
    json.dump(videos, f, ensure_ascii=False, indent=4)

print("Données sauvegardées ")
print(f"Nombre de vidéos : {len(videos)}")